In [ ]:
#| default_exp game/mechanics/time

In [ ]:
#| export
from __future__ import annotations
import sys
import math
from fastcore.basics import patch
from fasthtml.common import *
from fasthtml.jupyter import *
import pandas as pd
import numpy as np
from dataclasses import dataclass, field
from collections import deque , Counter

from monsterui.all import TableT, Card, CardHeader, CardTitle

In [ ]:
from HexMagic.game.data import Settlement, Kingdom, Piece,  GameBoard

In [ ]:
#| export

from dataclasses import dataclass, field
from enum import Enum
from typing import Literal
import heapq, math

import pyomo.environ as pyo
from fasthtml.common import *
from HexMagic.primitives import HexGrid, HexPosition
from HexMagic.game.data import Piece, PieceType, GameBoard

In [ ]:
#| export
import httpx
import random
import pandas as pd

from dataclasses import dataclass
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import math

from dataclasses import dataclass, field
from functools import cached_property

import heapq
import io
from monsterui.all import *
from scipy.optimize import linear_sum_assignment
from scipy.optimize import milp, LinearConstraint, Bounds

In [ ]:
from HexMagic.styles import StyleCSS,  SVGBuilder, SVGDef
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord ,HexDragMap, HexTouchMap, HexRegion
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import FlowOverlay, CreamOverlay, RiverOverlay, ClimateOverlay, OverlayContext
from HexMagic.game.food import FoodOverlay, FoodYield, PieceOverlay, PieceDataOverlay, PieceArrowOverlay, SquadSymbolOverlay

In [ ]:
from HexMagic.game.mechanics.simulator import TurnReport, SimEventType, FoodSimulator, squad_chart, SquadFoodTrailOverlay, apply_food_profile
from HexMagic.game.mechanics.simulator import  knownWorld, smart_placement,optimal_camp_layout, PieceShadow, GameParts,FOOD_PROFILE_V2, _solver_ok
from HexMagic.game.piece import pieces_center, StatBar, SquadVisionOverlay, GameParts,SettlementOverlay, KingdomNamesOverlay, PieceOverlay
from HexMagic.game.piece import  _facing_toward, GroupedPieceList, PieceList
from HexMagic.game.flag import GameContext, DiagramGlyphs
from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame, CountryFlag
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList, Squad

In [ ]:
from HexMagic.game.mechanics.space import CrossingPlan, Corridor, Target, Assignment, CorridorType, pawn_covered_route, find_stations_v3, simple_hex_path, route_to_instructions, optimal_lighthouse_placement, food_redistribute_adjacent, CorridorOverlay

## Demo

In [ ]:
myStuff = GameParts()
def rebootWorld():
    myStuff.loadGeology(knownWorld())
myStuff.grid.adjustRadius(40)
rebootWorld()

In [ ]:
showDemo = True
myStuff.grid.adjustRadius(40)

In [ ]:
showDemo = False

In [ ]:
TerrainDisplay(
    TerrainOverlay(),
    SettlementOverlay(),
    SquadVisionOverlay([myStuff.queenGuard]),
    RiverOverlay(max_width=5),
    terrain=myStuff.terr,
    board=myStuff.board,
    basins=myStuff.basin,
    debug=not showDemo
    )

## Sim Adapter

In [ ]:
# ── Schedule schema + helpers ─────────────────────────────────────────────

SCHEDULE_COLUMNS = [
    'schedule_id', 'piece_id', 'piece_name', 'turn', 'hex', 'facing',
    'action', 'food', 'health', 'segment_id', 'segment_type',
    'corridor_id', 'note',
]

def _exact_facing(grid, from_hex, to_hex):
    "Exact facing from from_hex to adjacent to_hex, or None."
    dirs = HexPosition.directions()
    for d in range(6):
        if grid.hexposition_to_index(dirs[d], from_hex) == to_hex:
            return d
    return None

def empty_schedule(schedule_id='empty'):
    return pd.DataFrame(columns=SCHEDULE_COLUMNS)

def schedule_timetable(df, title='Schedule', max_cols=25):
    "Render a Schedule DataFrame as a pieces×turns timetable."
    if df.empty: return Card(P('Empty schedule'), header=CardTitle(title))

    pieces = df.piece_name.unique()
    turns  = sorted(df.turn.unique())
    if len(turns) > max_cols:
        step = max(1, len(turns) // max_cols)
        turns = turns[::step]

    header = Tr(Th('piece'), *[Th(f't{int(t)}') for t in turns])
    rows = []
    for pname in pieces:
        pdata = df[df.piece_name == pname].set_index('turn')
        cells = [Th(pname)]
        for t in turns:
            if t in pdata.index:
                r = pdata.loc[t]
                if isinstance(r, pd.DataFrame): r = r.iloc[0]
                h = int(r['hex']); f = int(r['facing']); fd = r['food']
                tip = f"f={f} food={fd:.1f} {r.get('action','—')}"
                cells.append(Td(f'⬡{h}', title=tip, style='font-size:0.8em'))
            else:
                cells.append(Td('—'))
        rows.append(Tr(*cells))

    return Card(
        Table(Thead(header), Tbody(*rows), cls=TableT.sm),
        header=CardTitle(title),
    )


In [ ]:
_EVENT_TO_ACTION = {
    'moved': 'FORWARD', 'bumped': 'BUMPED', 'harvested': 'HARVEST',
    'gave': 'GIVE', 'rotated': 'ROT', 'blocked': 'BLOCKED',
    'attacked': 'ATTACK', 'countered': 'COUNTER',
}
_ACTION_RANK = {
    'FORWARD': 0, 'BUMPED': 1, 'ATTACK': 2, 'COUNTER': 3,
    'HARVEST': 4, 'GIVE': 5, 'BLOCKED': 6, 'ROT': 7,
}

@dataclass
class SimAdapter:
    """Extracts actual schedules from FoodSimulator output.

    Sim snapshots (post-step state) are offset +1 so turn 0 = pre-sim.
    """
    sim:       FoodSimulator
    corridors: list[Corridor]
    _names:    dict = field(default_factory=dict)

    def __post_init__(self):
        self._names = {p.id: p.name for p in self.sim.pieces}

    def _primary_action(self, piece_id, sim_turn):
        "Most significant voluntary action for (piece, sim_turn)."
        df = self.sim.df
        evts = df.loc[(df.piece_id == piece_id) & (df.turn == sim_turn), 'event']
        best, best_r = '—', 999
        for ev in evts:
            act = _EVENT_TO_ACTION.get(ev)
            if act and _ACTION_RANK.get(act, 99) < best_r:
                best, best_r = act, _ACTION_RANK[act]
        return best

    def _map_loc(self, hex_idx):
        "Map hex → (corridor_label, segment_label, segment_type) or Nones."
        for cor in self.corridors:
            if hex_idx not in cor: continue
            w = cor.index(hex_idx)
            try:
                seg = cor[w]
                return cor.label, seg.label, seg.type
            except KeyError:
                return cor.label, None, None
        return None, None, None

    def _at_target(self, hex_idx):
        "Check if hex is a corridor target — returns purpose string or None."
        for cor in self.corridors:
            if cor.target and cor.target.hex == hex_idx:
                return cor.target.purpose
        return None

    def _has_event(self, pid, sim_turn, ename):
        df = self.sim.df
        return ((df.piece_id == pid) & (df.turn == sim_turn) & (df.event == ename)).any()

    def _row(self, pid, turn, hex_idx, facing, action, food, health,
             schedule_id, note=''):
        cid, slabel, stype = self._map_loc(hex_idx)
        tgt = self._at_target(hex_idx)
        if tgt and not note:
            note = f'🎯 {tgt}'
        return dict(
            schedule_id=schedule_id, piece_id=pid,
            piece_name=self._names.get(pid, pid[:8]),
            turn=turn, hex=hex_idx, facing=facing,
            action=action, food=round(food, 1),
            health=round(health, 1),
            segment_id=slabel, segment_type=stype,
            corridor_id=cid, note=note)

    def extract(self, schedule_id='actual', initial_states=None, piece_ids=None):
        """Build Schedule DataFrame."""
        ids  = piece_ids or set(self._names)
        rows = []

        # Turn 0 from initial_states
        if initial_states:
            for pid, st in initial_states.items():
                if pid not in ids: continue
                rows.append(self._row(
                    pid, 0, st['location'], st['facing'],
                    '—', st['food'], st.get('health', 100),
                    schedule_id, 'start'))

        # Snapshots → turn 1, 2, ...
        for snap in self.sim.snapshots:
            t = snap['turn'] + 1
            for pid, state in snap['piece_states'].items():
                if pid not in ids: continue
                act = self._primary_action(pid, snap['turn'])
                note = ''
                if self._has_event(pid, snap['turn'], 'died'):      note = '💀 dead'
                elif self._has_event(pid, snap['turn'], 'starved'): note = 'starving'
                rows.append(self._row(
                    pid, t, state['location'], state['facing'],
                    act, state['food'], state['health'],
                    schedule_id, note))

        return pd.DataFrame(rows, columns=SCHEDULE_COLUMNS) if rows else empty_schedule()



In [ ]:
@dataclass
class VarianceReport:
    "Joins budget + actual schedules and flags divergence."
    budget: pd.DataFrame
    actual: pd.DataFrame

    def compute(self):
        v = pd.merge(self.budget, self.actual,
                     on=['piece_id', 'turn'], suffixes=('_bgt', '_act'),
                     how='outer')
        v['hex_match']    = v.hex_bgt == v.hex_act
        v['facing_match'] = v.facing_bgt == v.facing_act
        v['food_delta']   = v.food_act - v.food_bgt

        def _status(r):
            if pd.notna(r.get('note_act')) and '💀' in str(r.note_act): return '💀 dead'
            if pd.isna(r.get('hex_act')): return 'not_started'
            if pd.isna(r.get('hex_bgt')): return 'unplanned'
            if r.hex_match: return 'on_track'
            return 'off_route'

        v['status'] = v.apply(_status, axis=1)
        return v

    def summary(self):
        v  = self.compute()
        nm = v.piece_name_bgt.fillna(v.piece_name_act)
        out = []
        for pname in nm.unique():
            pv = v[nm == pname]
            on = (pv.status == 'on_track').sum()
            fd = pv.food_delta.mean()
            worst = 'on_track'
            for s in ['💀 dead', 'off_route', 'not_started']:
                if (pv.status == s).any(): worst = s; break
            out.append(dict(piece=pname, on_track=f'{on}/{len(pv)}',
                            avg_food_delta=f'{fd:+.1f}' if pd.notna(fd) else '—',
                            status=worst))
        return pd.DataFrame(out)

    def __ft__(self):
        v  = self.compute()
        nm = v.piece_name_bgt.fillna(v.piece_name_act)
        pieces = nm.unique()
        turns  = sorted(v.turn.unique())

        _col = {'on_track': '#d4edda', 'off_route': '#f8d7da',
                '💀 dead': '#555', 'unplanned': '#fff3cd',
                'not_started': '#e2e3e5'}

        header = Tr(Th('piece'), *[Th(f't{int(t)}') for t in turns])
        trows  = []
        for pname in pieces:
            pv = v[nm == pname].set_index('turn')
            cells = [Th(pname)]
            for t in turns:
                if t in pv.index:
                    r = pv.loc[t]
                    if isinstance(r, pd.DataFrame): r = r.iloc[0]
                    st  = r['status']
                    bg  = _col.get(st, '#fff')
                    ha  = r.get('hex_act', r.get('hex_bgt'))
                    hb  = r.get('hex_bgt')
                    hs  = f'⬡{int(ha)}' if pd.notna(ha) else '—'
                    fd  = r.get('food_delta', 0)
                    tip = (f"bgt=⬡{int(hb) if pd.notna(hb) else '?'} "
                           f"{st} food:{fd:+.1f}" if pd.notna(fd) else st)
                    cells.append(Td(hs, title=tip,
                                    style=f'background:{bg};font-size:0.8em'))
                else:
                    cells.append(Td('—', style='background:#f5f5f5'))
            trows.append(Tr(*cells))

        timetable = Table(Thead(header), Tbody(*trows), cls=TableT.sm)

        # Summary table
        s = self.summary()
        sh = list(s.columns)
        sr = [Tr(*[Td(str(row[c])) for c in sh]) for _, row in s.iterrows()]
        summ = Table(Thead(Tr(*[Th(h) for h in sh])), Tbody(*sr), cls=TableT.sm)

        return Div(
            Style("td[title]{cursor:help}"),
            Card(timetable, header=CardTitle('Variance Timetable')),
            Card(summ,      header=CardTitle('Summary')),
        )


# 10. QueenCoverage

The main planning object that owns corridors, targets, and assignments. Provides the high-level API: `build_corridor`, `add_target`, `propose_crossing`, `plan`.

In [ ]:
@dataclass
class CampPlan:
    """Budget allocation for pawn roles along a corridor."""
    origin:    list[tuple[int, int]] = field(default_factory=list)  # (hex, tier)
    dest:      list[tuple[int, int]] = field(default_factory=list)
    station:   list[int]             = field(default_factory=list)  # pawn hexes
    target:    Target = None
    n_origin:  int = 0
    n_dest:    int = 0
    n_station: int = 0

    @property
    def total_pawns(self): return self.n_origin + self.n_dest + self.n_station

    def summary(self):
        return (f"{self.n_origin} origin + {self.n_station} station "
                f"+ {self.n_dest} advance = {self.total_pawns} pawns")


In [ ]:
@dataclass
class RosterResult:
    """Output from a shadow-building step."""
    shadows: list       = field(default_factory=list)
    entries: list[dict] = field(default_factory=list)
    timing:  int        = 0  # turns until this group is ready


In [ ]:
@dataclass
class QueenCoverage:
    "Logistics planning layer: corridors, targets, and queen assignments."
    board:      GameBoard
    grid:       object        # HexGrid
    elevations: np.ndarray
    food_tiers: np.ndarray
    profile:    dict          # FOOD_PROFILE_V2

    corridors:   list[Corridor]   = field(default_factory=list)
    targets:     list[Target]     = field(default_factory=list)
    queens:      list[Piece]      = field(default_factory=list)
    assignments: list[Assignment] = field(default_factory=list)

    # ── Lookups ──────────────────────────────────────────────────

    def _countries(self):
        return getattr(self.board, 'terrain', None) and \
               self.board.terrain.fields.get('country')

    def find_corridor(self, start: int, end: int) -> Corridor | None:
        return next((c for c in self.corridors
                     if c.start == start and c.end == end), None)

    def food_level(self, hex: int) -> float:
        tier = int(self.food_tiers[hex]) if 0 <= hex < len(self.food_tiers) else 0
        p = self.profile
        return max(0.0, p.get('pawn_harvest', 1.3) * tier - p.get('pawn_diet', 0.7))

    # ── Targets ──────────────────────────────────────────────────

    def add_target(self, hex: int, purpose: str,
                   priority: float = 1.0, deadline: int | None = None,
                   payoff: float = 0.0, style: StyleCSS = None,
                   glyph: object = None) -> Target:
        t = Target(hex=hex, purpose=purpose, priority=priority,
                   deadline=deadline, payoff=payoff,
                   style=style, glyph=glyph)
        self.targets.append(t)
        return t

    # ── Shadow factories ─────────────────────────────────────────

    def _make_pawn_shadow(self, hex, facing, name,
                          instructions, patrol=True,
                          owner_id=None, flag=None) -> PieceShadow:
        p = self.profile
        ps = PieceShadow(
            id=name, name=name,
            piece_type=PieceType.PAWN, birth_year=0,
            location=hex, facing=facing,
            food=p['pawn_capacity'], food_capacity=p['pawn_capacity'],
            diet=p['pawn_diet'], health=100, max_health=100,
            move_strength=2, harvest_strength=p['pawn_harvest'],
            sight=p['pawn_sight'],
            owner_id=owner_id, flag=flag)
        ps.instructions = InstructionList(instructions, cursor=0, patrol=patrol)
        return ps

    def _make_queen_shadow(self, queen: Piece, route: list[int],
                           wait_turns: int) -> PieceShadow:
        qs = PieceShadow.from_piece(queen)
        qs.location = route[0]
        qs.food = self.profile['queen_capacity']
        wait = [Instruction.PAUSE.value] * wait_turns
        march, _ = route_to_instructions(route, self.grid, qs.facing)
        qs.instructions = InstructionList(wait + march, cursor=0, patrol=False)
        return qs

    # ── Camp planning ────────────────────────────────────────────

    def _find_camp_hexes(self, center: int, n: int = 2,
                         max_ring: int = 3,
                         exclude: set = None) -> list[tuple[int, int]]:
        """Find best food hexes near a center. Returns (hex, tier) pairs."""
        exclude = exclude or set()
        grid, elevs, tiers = self.grid, self.elevations, self.food_tiers
        cands = []
        for ring in range(1, max_ring + 1):
            for hp in HexPosition.origin().ring(ring):
                idx = grid.hexposition_to_index(hp, center)
                if idx < 0 or idx in grid.invalidRegion: continue
                if idx >= len(elevs) or elevs[idx] <= 0: continue
                if idx in exclude: continue
                tier = int(tiers[idx]) if idx < len(tiers) else 0
                if tier > 0:
                    cands.append((idx, tier))
        cands.sort(key=lambda x: -x[1])
        return cands[:n]

    def _plan_camps(self, corridor: Corridor, budget: int) -> CampPlan:
        """Allocate pawn budget across origin, station, and dest roles."""
        route_set = set(corridor.hexes)
        target = corridor.target

        origin = self._find_camp_hexes(corridor.start, exclude=route_set)
        dest   = self._find_camp_hexes(corridor.end, exclude=route_set)

        station_hexes = [h for seg in corridor.segments
                         if seg.type == 'station' for h in seg.hexes]

        # Purpose-aware weighting
        if target and target.purpose == 'attack':
            max_dest = min(len(dest), 3)
            max_origin = 1
        elif target and target.purpose == 'harvest':
            max_dest = min(len(dest), 1)
            max_origin = 1
        else:
            max_dest = min(len(dest), 2)
            max_origin = min(len(origin), 2)

        n_origin  = min(max_origin, budget)
        n_dest    = min(max_dest, budget - n_origin)
        n_station = min(len(station_hexes), budget - n_origin - n_dest)

        return CampPlan(
            origin=origin[:n_origin], dest=dest[:n_dest],
            station=station_hexes[:n_station], target=target,
            n_origin=n_origin, n_dest=n_dest, n_station=n_station)

    # ── Roster builders (one per role) ───────────────────────────

    def _build_origin_shadows(self, camp: CampPlan,
                              queen: Piece, route: list[int]) -> RosterResult:
        """Origin camp pawns: park near start, HARVEST+GIVE in place."""
        result = RosterResult()
        harvest_give = [Instruction.HARVEST.value, Instruction.GIVE.value]

        for i, (hx, tier) in enumerate(camp.origin):
            facing = _facing_toward(self.grid, hx, route[0])
            ps = self._make_pawn_shadow(
                hx, facing, f'camp_o_{i}', harvest_give,
                patrol=True, owner_id=queen.owner_id, flag=queen.flag)
            result.shadows.append(ps)
            result.entries.append(dict(
                piece_type=PieceType.PAWN, hex=hx,
                facing=facing, role='origin_camp'))
        return result

    def _build_advance_shadows(self, camp: CampPlan,
                               queen: Piece,
                               route: list[int]) -> RosterResult:
        """Advance party: sprint to dest camp hexes, then park + harvest."""
        result = RosterResult()
        harvest_give = [Instruction.HARVEST.value, Instruction.GIVE.value]

        for i, (hx, tier) in enumerate(camp.dest):
            sprint_path = simple_hex_path(route[0], hx, self.grid, self.elevations)
            if not sprint_path: continue

            sprint_rules, end_facing = route_to_instructions(
                sprint_path, self.grid, 0)

            want_f = _facing_toward(self.grid, hx, route[-1])
            diff = (want_f - end_facing) % 6
            rot = ([Instruction.ROT_R.value] * diff if diff <= 3
                   else [Instruction.ROT_L.value] * (6 - diff))

            park = harvest_give * 80
            instructions = sprint_rules + rot + park

            ps = self._make_pawn_shadow(
                route[0], 0, f'camp_d_{i}', instructions,
                patrol=False, owner_id=queen.owner_id, flag=queen.flag)
            result.shadows.append(ps)
            result.entries.append(dict(
                piece_type=PieceType.PAWN, hex=hx,
                facing=want_f, role='advance_party'))

            arr = math.ceil(len(sprint_path) / 2) + len(rot) + 2
            result.timing = max(result.timing, arr)

        return result

    def _build_station_shadows(self, camp: CampPlan,
                               corridor: Corridor,
                               queen: Piece) -> RosterResult:
        """Mid-route station pawns: park at station hexes, HARVEST+GIVE."""
        result = RosterResult()
        route = corridor.hexes
        harvest_give = [Instruction.HARVEST.value, Instruction.GIVE.value]
        pawn_n = 0

        for seg in corridor.segments:
            if seg.type != 'station': continue
            mid_w = (seg.route_start + seg.route_end) // 2
            for hx in seg.hexes:
                if pawn_n >= camp.n_station: break
                facing = _facing_toward(self.grid, hx, route[mid_w])
                ps = self._make_pawn_shadow(
                    hx, facing, f'station_{pawn_n}', harvest_give,
                    patrol=True, owner_id=queen.owner_id, flag=queen.flag)
                result.shadows.append(ps)
                result.entries.append(dict(
                    piece_type=PieceType.PAWN, hex=hx,
                    facing=facing, role='station'))
                pawn_n += 1
        return result

    def _build_bishop_shadows(self, corridor: Corridor,
                              queen: Piece,
                              n_bishops: int) -> RosterResult:
        """Bishop placement: lighthouse MIP first, escort fallback."""
        result = RosterResult()
        route = corridor.hexes

        all_gaps = []
        for seg in corridor.segments:
            if seg.type == 'caravan':
                all_gaps.extend(range(seg.route_start, seg.route_end + 1))

        if not all_gaps or n_bishops <= 0:
            return result

        # Try lighthouse placement
        lh = optimal_lighthouse_placement(
            all_gaps, route, self.grid, self.elevations,
            n_bishops=n_bishops, profile=self.profile)

        if lh and lh['feasible']:
            for pl in lh['placements']:
                bs, arr = self._make_bishop_shadow(
                    pl['hex'], pl['serves_route_w'], route,
                    self.profile, queen,
                    f'blh_{pl["bishop"]}', f'Bishop_{pl["bishop"]}')
                if bs:
                    result.shadows.append(bs)
                    result.entries.append(dict(
                        piece_type=PieceType.BISHOP, hex=pl['hex'],
                        facing=bs.facing, role='lighthouse'))
                    result.timing = max(result.timing, arr)
        else:
            # Escort fallback
            gap_segs = _split_contiguous(all_gaps)
            for bi, gseg in enumerate(gap_segs[:n_bishops]):
                mid_w = gseg[len(gseg) // 2]
                park_hex = route[mid_w]
                bs, arr = self._make_bishop_shadow(
                    park_hex, gseg, route, self.profile, queen,
                    f'besc_{bi}', f'Escort_{bi}')
                if bs:
                    result.shadows.append(bs)
                    result.entries.append(dict(
                        piece_type=PieceType.BISHOP, hex=park_hex,
                        facing=bs.facing, role='escort'))
                    result.timing = max(result.timing, arr)

        return result

    # ── Corridor metrics ─────────────────────────────────────────

    def _metrics_from_segments(self, segments: list[Segment]) -> dict:
        b_cap = self.profile.get('bishop_capacity', 140.0)
        if not segments:
            return dict(n_stations=0, n_gaps=0, fuel_needed=0.0,
                        bishop_min_fuel=0.0, duration=0, valid=True)

        n_stations = sum(1 for s in segments if s.type == 'station')
        gap_segs   = [s for s in segments if s.type == 'caravan']
        n_gaps     = sum(s.gap_len for s in gap_segs)
        fuel_needed = sum(s.fuel_needed for s in gap_segs)
        duration = int(b_cap / max(fuel_needed, 0.1)) if fuel_needed > 0 else 0

        return dict(n_stations=n_stations, n_gaps=n_gaps,
                    fuel_needed=fuel_needed, bishop_min_fuel=b_cap,
                    duration=duration, valid=True)

    # ── Corridor construction ────────────────────────────────────

    def build_corridor(self, start: int, end: int,
                       kind: CorridorType = CorridorType.TEMPORARY,
                       n_pawns: int = 4, n_bishops: int = 1,
                       target: Target = None,
                       style: StyleCSS = None,
                       glyph: DiagramGlyphs = None) -> Corridor | None:
        existing = self.find_corridor(start, end)
        if existing:
            print(f"build_corridor: reusing ⬡{start}→⬡{end}")
            return existing

        p = self.profile
        pcr = pawn_covered_route(
            start, end, grid=self.grid, elevations=self.elevations,
            food_tiers=self.food_tiers,
            pawn_harvest=p.get('pawn_harvest', 1.3),
            pawn_diet=p.get('pawn_diet', 0.7),
            queen_diet=p.get('queen_diet', 7.0),
            cone_range=p.get('pawn_sight', 4),
            n_bishops=1,
            bishop_capacity=p.get('bishop_capacity', 140.0),
            bishop_diet=p.get('bishop_diet', 2.5),
            countries=self._countries())
        if pcr is None:
            print(f"build_corridor: no route ⬡{start}→⬡{end}")
            return None

        segments = find_stations_v3(pcr, n_pawns=n_pawns, n_bishops=1, profile=p)
        metrics = self._metrics_from_segments(segments)

        if n_bishops > 1 and metrics['fuel_needed'] > 0:
            metrics['bishop_min_fuel'] = p.get('bishop_capacity', 140.0) * n_bishops
            metrics['duration'] = int(metrics['bishop_min_fuel'] /
                                      max(metrics['fuel_needed'], 0.1))

        corridor = Corridor(hexes=pcr['route'], kind=kind,
                            segments=segments, target=target,
                            style=style, glyph=glyph, **metrics)
        self.corridors.append(corridor)

        tag = '✅' if corridor.feasible else '❌'
        print(f"Corridor ⬡{start}→⬡{end} ({kind.value}): "
              f"{corridor.length}hex, {corridor.n_stations} stations, "
              f"{corridor.n_gaps} gaps, fuel={corridor.fuel_needed:.1f} {tag}")
        return corridor

    # ── Bishop / congestion helpers ──────────────────────────────

    def assign_bishop(self, corridor: Corridor, bishop: Piece):
        fuel = getattr(bishop, 'food', self.profile.get('bishop_capacity', 140.0))
        corridor.bishop_min_fuel = min(corridor.bishop_min_fuel, fuel)

    def update_congestion(self):
        for c in self.corridors:
            rs = set(c.hexes)
            c.congestion = sum(
                1 for kingdom in self.board.kingdoms
                for settlement in kingdom.settlements
                for piece in settlement.citizens
                if piece.location in rs)

    # ── Crossing proposal ────────────────────────────────────────

    def propose_crossing(self, corridor: Corridor, queen: Piece,
                         available_pawns: int = 5,
                         available_bishops: int = 2,
                         max_retries: int = 3,
                         n_turns: int = 60) -> CrossingPlan:
        """Propose a validated crossing: assemble roster, simulate, retry."""
        camp = self._plan_camps(corridor, available_pawns)
        print(f"  Camp plan: {camp.summary()}")
        print(f"  Origin: {camp.origin}  Dest: {camp.dest}")

        best_plan = None
        route = corridor.hexes

        for attempt in range(max_retries + 1):
            extra_wait = attempt * 2

            # ── Assemble roster ──
            origin_r  = self._build_origin_shadows(camp, queen, route)
            advance_r = self._build_advance_shadows(camp, queen, route)
            station_r = self._build_station_shadows(camp, corridor, queen)
            bishop_r  = self._build_bishop_shadows(corridor, queen, available_bishops)

            queen_start = max(bishop_r.timing, advance_r.timing) + extra_wait
            qs = self._make_queen_shadow(queen, route, queen_start)

            shadows = ([qs] + origin_r.shadows + advance_r.shadows
                       + station_r.shadows + bishop_r.shadows)
            roster = ([dict(piece_type=PieceType.QUEEN, hex=route[0],
                            facing=qs.facing, role='queen')]
                      + origin_r.entries + advance_r.entries
                      + station_r.entries + bishop_r.entries)

            # ── Stamp ownership ──
            for s in shadows:
                s.owner_id = queen.owner_id
                s.flag = queen.flag

            # ── Simulate ──
            plan = CrossingPlan(
                corridor=corridor, roster=roster,
                queen_start_turn=queen_start)
            plan.simulate(
                shadows=shadows, queen_shadow=qs,
                grid=self.grid, elevations=self.elevations,
                food_tiers=self.food_tiers, profile=self.profile,
                countries=self._countries(), n_turns=n_turns)

            best_plan = plan

            if plan.feasible:
                roles = Counter(r['role'] for r in roster)
                print(f"✅ Crossing (attempt {attempt}): roles={dict(roles)}, "
                      f"start=t{queen_start}, arrive=t{plan.queen_arrival}, "
                      f"min_food={plan.food_margin:.1f}")
                return plan

            print(f"❌ Attempt {attempt}: arrive={'yes' if plan.queen_arrival < n_turns else 'no'}, "
                  f"min_food={plan.food_margin:.1f}, wait={queen_start}t")

        print(f"⚠️ All {max_retries + 1} attempts failed")
        return best_plan


In [ ]:
def _split_contiguous(indices):
    """Split a sorted list of ints into contiguous runs."""
    if not indices: return []
    segs, cur = [], [indices[0]]
    for g in indices[1:]:
        if g == cur[-1] + 1: cur.append(g)
        else: segs.append(cur); cur = [g]
    segs.append(cur)
    return segs


@patch
def _make_bishop_shadow(self: QueenCoverage, park_hex, served_ws, route,
                        profile, queen, shadow_id, shadow_name):
    """Create a bishop shadow that sprints to park_hex, faces the gap, parks.

    Returns (PieceShadow, arrival_turn) or (None, 0) if no path.
    """
    bp = simple_hex_path(route[0], park_hex, self.grid, self.elevations)
    if not bp: return None, 0

    sprint, end_f = route_to_instructions(bp, self.grid, 0)

    # Face toward the middle of the served gap for max sight coverage
    mid_w = served_ws[len(served_ws) // 2]
    want_f = _facing_toward(self.grid, park_hex, route[mid_w])
    d = (want_f - end_f) % 6
    rot = ([Instruction.ROT_R.value] * d if d <= 3
           else [Instruction.ROT_L.value] * (6 - d))

    park = [Instruction.HARVEST.value, Instruction.GIVE.value] * 60

    bs = PieceShadow(
        id=shadow_id, name=shadow_name,
        piece_type=PieceType.BISHOP, birth_year=0,
        location=route[0], facing=0,
        food=profile['bishop_capacity'],
        food_capacity=profile['bishop_capacity'],
        diet=profile.get('bishop_diet', 3.5),
        health=100, max_health=100, move_strength=3,
        harvest_strength=profile.get('bishop_harvest', 0.7),
        sight=profile['pawn_sight'],
        owner_id=queen.owner_id, flag=queen.flag)
    bs.instructions = InstructionList(sprint + rot + park, cursor=0, patrol=False)

    arrival = math.ceil(len(bp) / 3) + 2  # bishop speed 3, +safety
    return bs, arrival

In [ ]:
@patch
def simulate(self: CrossingPlan,
             shadows: list, queen_shadow,
             grid, elevations, food_tiers,
             profile: dict, countries=None,
             n_turns: int = 60) -> 'CrossingPlan':
    """Run FoodSimulator on shadows, populate result fields."""
    route = self.corridor.hexes

    shadow_squad = Squad.squads(1, queen_shadow.flag)[0]
    shadow_squad.pieces = list(shadows)

    initial = {s.id: dict(location=s.location, facing=s.facing,
                           food=s.food, health=s.health)
               for s in shadows}

    sim = FoodSimulator(
        grid=grid, elevations=elevations,
        food_tiers=food_tiers, pieces=shadows,
        countries=countries,
        squads=[shadow_squad],
        callbacks={shadow_squad.id: food_redistribute_adjacent})
    sim.run(num_turns=n_turns)

    # ── Evaluate queen trajectory ──
    q_snap = [s['piece_states'].get(queen_shadow.id) for s in sim.snapshots]
    alive   = q_snap[-1] and q_snap[-1]['health'] > 0 if q_snap else False
    arrived = any(s and s['location'] == route[-1] for s in q_snap)
    min_food = min((s['food'] for s in q_snap if s and s['health'] > 0),
                   default=0)

    arrival_turn = n_turns
    for i, s in enumerate(q_snap):
        if s and s['location'] == route[-1]:
            arrival_turn = i + 1
            break

    bishop_fuel = 0
    if sim.snapshots:
        for s in shadows:
            if s.piece_type == PieceType.BISHOP:
                ls = sim.snapshots[-1]['piece_states'].get(s.id)
                if ls:
                    bishop_fuel += profile['bishop_capacity'] - ls.get('food', 0)

    adapter = SimAdapter(sim, corridors=[self.corridor])
    self.schedule = adapter.extract(schedule_id=f'crossing',
                                     initial_states=initial)

    self.feasible = arrived and alive
    self.queen_arrival = arrival_turn
    self.food_margin = min_food
    self.bishop_fuel_used = bishop_fuel
    return self


In [ ]:
rebootWorld()
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

cities = []
for country in myStuff.board.kingdoms:
    for city in country.settlements:
        cities.append(city)

c0, c1 = cities[0].location, cities[1].location
print(f"Cities: ⬡{c0} → ⬡{c1}\n")

# 2. Build coverage + corridor
qc = QueenCoverage(
    board=myStuff.board, grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers, profile=FOOD_PROFILE_V2,
    queens=[myStuff.queen],
)

corridor = qc.build_corridor(c0, c1, kind=CorridorType.PERMANENT,
                              n_pawns=5, n_bishops=2)
print(f"\nCorridor: {corridor.length} hexes, "
      f"{corridor.n_stations} stations, {corridor.n_gaps} gaps, "
      f"feasible={'✅' if corridor.feasible else '❌'}")

if not corridor.feasible:
    print("❌ Corridor not feasible — can't test crossing")
else:
    # 3. Teleport queen to corridor start, fill her up
    queen = myStuff.queen
    queen.location = corridor.hexes[0]
    apply_food_profile([queen])
    print(f"\nQueen {queen.name} at ⬡{queen.location}, "
          f"food={queen.food:.1f}, diet={queen.diet}")

    # 4. Propose crossing
    print(f"\n{'='*60}")
    print("Proposing crossing plan...")
    plan = qc.propose_crossing(
        corridor, queen,
        available_pawns=5,
        available_bishops=2,
        n_turns=60,
        max_retries=3,
    )

    # 5. Results
    print(f"\n{'='*60}")
    print(f"Feasible:     {plan.feasible}")
    print(f"Pieces:       {dict(plan.pieces_needed())}")
    print(f"Queen starts: turn {plan.queen_start_turn}")
    print(f"Arrival:      turn {plan.queen_arrival}")
    print(f"Food margin:  {plan.food_margin:.1f}")
    print(f"Bishop fuel:  {plan.bishop_fuel_used:.0f}")

    # 6. Show the plan card + timetable
    show(plan)

In [ ]:

ctx = myStuff.overlayContext()

TerrainDisplay(
    SettlementOverlay(),
    CorridorOverlay(),
    terrain=myStuff.terr,
    board=myStuff.board,
    basins=myStuff.basin,
    corridors=[corridor],
    context_cls=GameContext,
    debug = not showDemo
)

Lets do corridor route and piece starting positions.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import RegularPolygon
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe

grid = myStuff.grid
elevs = myStuff.terr.elevations
ncols = 20
R = 40
sqrt3 = math.sqrt(3)

def hex_xy(idx):
    """Offset → pixel center (pointy-top, odd-r)."""
    r, c = divmod(idx, ncols)
    return sqrt3 * R * (c + 0.5 * (r % 2)),  1.5 * R * r

# ── Auto-zoom to corridor region ──
cor_pts = [hex_xy(h) for h in corridor.hexes]
roster_pts = [hex_xy(e['hex']) for e in plan.roster]
all_pts = cor_pts + roster_pts
all_xs, all_ys = zip(*all_pts)
pad = R * 8
xlim = (min(all_xs) - pad, max(all_xs) + pad)
ylim = (min(all_ys) - pad, max(all_ys) + pad)

fig, ax = plt.subplots(figsize=(16, 11))

# 1. Terrain background (faded, only visible hexes)
elev_pos = [e for e in elevs if e > 0]
emax = max(elev_pos) if elev_pos else 1
for idx in range(len(elevs)):
    x, y = hex_xy(idx)
    if not (xlim[0] <= x <= xlim[1] and ylim[0] <= y <= ylim[1]): continue
    color = '#6baed6' if elevs[idx] <= 0 else plt.cm.YlGn(0.3 + 0.5*(1 - elevs[idx]/emax))
    ax.add_patch(RegularPolygon(
        (x, y), 6, radius=R*.97, orientation=math.pi/6,
        facecolor=color, edgecolor='#ddd', linewidth=.3, alpha=.45))

# 2. Corridor hexes by segment type
seg_colors = {'station': '#27ae60', 'caravan': '#e74c3c'}
for seg in corridor.segments:
    sc = seg_colors.get(seg.type, '#888')
    for w in range(seg.route_start, seg.route_end + 1):
        hx = corridor.hexes[w]
        x, y = hex_xy(hx)
        ax.add_patch(RegularPolygon(
            (x, y), 6, radius=R*.92, orientation=math.pi/6,
            facecolor=sc, edgecolor='white', linewidth=1.5, alpha=.75))
        ax.text(x, y + R*.32, str(hx), fontsize=5.5, ha='center',
                va='top', color='white', fontweight='bold')

# Direction arrows along corridor
for i in range(len(corridor.hexes) - 1):
    x1, y1 = hex_xy(corridor.hexes[i])
    x2, y2 = hex_xy(corridor.hexes[i+1])
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='white', lw=1.3, alpha=.6))

# 3. Roster pieces
role_style = {
    'queen':         ('♛', 22, '#9b59b6'),
    'origin_camp':   ('♟', 17, '#1a8c4e'),
    'advance_party': ('♟', 17, '#d35400'),
    'station':       ('♟', 17, '#2980b9'),
    'lighthouse':    ('♝', 19, '#8e44ad'),
}
for entry in plan.roster:
    x, y = hex_xy(entry['hex'])
    glyph, sz, clr = role_style.get(entry['role'], ('?', 14, '#333'))
    ax.text(x, y - R*.08, glyph, fontsize=sz, ha='center', va='center',
            color=clr, zorder=10,
            path_effects=[pe.withStroke(linewidth=4, foreground='white')])
    ax.text(x, y + R*.45, entry['role'].replace('_', ' '),
            fontsize=5, ha='center', color=clr, alpha=.8, zorder=10)

# 4. City markers
for city in cities:
    x, y = hex_xy(city.location)
    ax.plot(x, y, 's', color='gold', ms=14,
            markeredgecolor='#333', markeredgewidth=2, zorder=11)
    ax.text(x, y - R*.65, city.name, fontsize=9, ha='center',
            fontweight='bold', color='#2c3e50',
            bbox=dict(boxstyle='round,pad=.2', fc='white', alpha=.85, ec='none'))

# Legend
legs = [
    mpatches.Patch(color='#27ae60', alpha=.75, label='Station segment'),
    mpatches.Patch(color='#e74c3c', alpha=.75, label='Caravan (gap)'),
    plt.Line2D([0],[0], marker='s', color='w', markerfacecolor='gold',
               ms=10, markeredgecolor='#333', label='City'),
]
for role, (g, s, c) in role_style.items():
    legs.append(plt.Line2D([0],[0], marker=f'${g}$', color='w',
                           markerfacecolor=c, ms=13,
                           label=role.replace('_',' ').title()))
ax.legend(handles=legs, loc='upper left', fontsize=8, framealpha=.9)

ax.set_xlim(*xlim); ax.set_ylim(*ylim)
ax.invert_yaxis(); ax.set_aspect('equal'); ax.axis('off')
ax.set_title(
    f'Crossing Plan: ⬡{corridor.hexes[0]} → ⬡{corridor.hexes[-1]}\n'
    f'Queen starts t{plan.queen_start_turn} · arrives t{plan.queen_arrival} · '
    f'food margin {plan.food_margin:.1f}',
    fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
from HexMagic.game.data import PieceOverlay

In [ ]:
from HexMagic.game.mechanics.space import CorridorOverlay

In [ ]:
ctx = myStuff.overlayContext()

TerrainDisplay(
    SettlementOverlay(),
    CorridorOverlay(),
    terrain=myStuff.terr,
    board=myStuff.board,
    basins=myStuff.basin,
    corridors=[corridor],
    context_cls=GameContext,
    debug = not showDemo
)

I am wondering about a different kind of schedule where we have the hex locations at the top time as the rows and then who is there as the fields. we would want a sensible ordering of the columns so that there is some sort of adjacency (though it will have to break since we are projecting 2d space onto 1d)

In [ ]:
#| export
# ── Role config: emoji + color ──
_ROLE_CFG = {
    'queen':         ('👑', '#8e44ad', 'Queen'),
    'origin_camp':   ('🌾', '#27ae60', 'Origin Camp'),
    'advance_party': ('🏃', '#2980b9', 'Advance Party'),
    'station':       ('⛺', '#f39c12', 'Station'),
    'lighthouse':    ('🔦', '#c0392b', 'Lighthouse Bishop'),
    'escort':        ('🐫', '#e74c3c', 'Escort Bishop'),
}

def _role_legend():
    """Horizontal legend strip for space-time timetable."""
    items = []
    for role, (emoji, color, label) in _ROLE_CFG.items():
        items.append(Span(
            Span(emoji, style='font-size:1.1em'),
            Span(f' {label}', style=f'color:{color};font-weight:bold;font-size:0.8em'),
            style='margin-right:1em;white-space:nowrap',
        ))
    return Div(*items, style='display:flex;flex-wrap:wrap;gap:0.25em 0;padding:0.5em 0',
               cls='uk-text-muted')


def space_time_timetable(df, corridor=None, grid=None,
                         roster=None, title='Space-Time', max_rows=40):
    """Time×Space schedule: rows=turns, columns=hex locations, cells=occupants.
    
    Args:
        roster: list of dicts from CrossingPlan.roster — used for role→emoji mapping.
    """
    if df.empty:
        return Card(P('Empty schedule'), header=CardTitle(title))

    turns = sorted(df.turn.unique())
    if len(turns) > max_rows:
        step = max(1, len(turns) // max_rows)
        turns = turns[::step]

    all_hexes = sorted(df.hex.unique())

    # ── Column ordering ──
    if corridor and grid:
        route_set = set(corridor.hexes)
        def _sort_key(h):
            if h in route_set:
                return (corridor.index(h), 0)
            hp_h = grid.index_to_hexposition(h)
            best_w, best_d = 0, 999
            for w, rh in enumerate(corridor.hexes):
                hp_r = grid.index_to_hexposition(rh)
                d = hp_h.distance(hp_r)
                if d < best_d:
                    best_d = d
                    best_w = w
            return (best_w, 1)
        all_hexes = sorted(all_hexes, key=_sort_key)
    elif corridor:
        route_set = set(corridor.hexes)
        on_route = [h for h in corridor.hexes if h in set(all_hexes)]
        off_route = sorted(h for h in all_hexes if h not in route_set)
        all_hexes = on_route + off_route

    # ── Build piece_id → (emoji, color) from roster ──
    piece_meta = {}
    if roster:
        # Roster entries have piece_type + role; match to schedule by position
        # Build name→role from roster naming convention
        role_names = {}
        queen_name = None
        for r in roster:
            role = r.get('role', '')
            if role == 'queen':
                queen_name = True  # first piece in schedule is queen
            # We'll match by piece_name patterns below
            
    # Fallback: infer from piece_name in schedule
    for _, row in df.drop_duplicates('piece_id').iterrows():
        name = str(row.get('piece_name', ''))
        pid = row.piece_id
        matched = False
        
        # Match patterns to roles
        patterns = [
            ('camp_o', 'origin_camp'),
            ('camp_d', 'advance_party'),
            ('origin', 'origin_camp'),
            ('advance', 'advance_party'),
            ('station', 'station'),
            ('blh_', 'lighthouse'),
            ('besc_', 'escort'),
            ('bishop', 'lighthouse'),
        ]
        for pat, role in patterns:
            if pat in name.lower():
                emoji, color, _ = _ROLE_CFG[role]
                piece_meta[pid] = (emoji, color)
                matched = True
                break
        
        if not matched:
            # Check if it's the queen (first piece, or name matches)
            if any(t in name.lower() for t in ('queen',)) or row.get('piece_name', '') not in [
                r.get('piece_name') for r in (roster or []) if r.get('role') != 'queen']:
                # Default: check if piece_type info available
                emoji, color, _ = _ROLE_CFG.get('queen', ('❓', '#555', '?'))
                piece_meta[pid] = (emoji, color)
    
    # Simpler: if roster provided, use index alignment
    if roster:
        piece_ids_ordered = list(df.drop_duplicates('piece_id').piece_id)
        # Roster order matches shadow order: queen first, then camps, etc.
        for i, r in enumerate(roster):
            if i < len(piece_ids_ordered):
                role = r.get('role', '')
                if role in _ROLE_CFG:
                    emoji, color, _ = _ROLE_CFG[role]
                    piece_meta[piece_ids_ordered[i]] = (emoji, color)

    # ── Header: hex labels ──
    route_set = set(corridor.hexes) if corridor else set()
    hex_headers = []
    for h in all_hexes:
        on = h in route_set
        style = 'font-weight:bold' if on else 'opacity:0.6;font-style:italic'
        hex_headers.append(Th(f'⬡{h}', style=f'{style};font-size:0.75em;writing-mode:vertical-lr'))

    header = Tr(Th('t', style='width:2em'), *hex_headers)

    # ── Rows: one per turn ──
    hex_to_col = {h: i for i, h in enumerate(all_hexes)}
    rows = []

    for t in turns:
        t_data = df[df.turn == t]
        cells = [''] * len(all_hexes)  # accumulate strings per cell

        for _, row in t_data.iterrows():
            h = row.hex
            if h not in hex_to_col: continue
            ci = hex_to_col[h]

            pid = row.piece_id
            emoji, color = piece_meta.get(pid, ('❓', '#555'))
            food = row.get('food', 0)
            action = row.get('action', '—')
            name = row.get('piece_name', '?')
            tip = f"{name} f={food:.0f} {action}"

            cells[ci] += f'<span title="{tip}" style="cursor:help">{emoji}</span>'

        row_cells = []
        for content in cells:
            if content:
                bg = '#fff8e1' if content.count('<span') > 1 else ''
                style = f'background:{bg};text-align:center' if bg else 'text-align:center'
                row_cells.append(Td(NotStr(content), style=style))
            else:
                row_cells.append(Td('', style='background:#fafafa'))

        rows.append(Tr(
            Td(Strong(f'{int(t)}'), style='font-size:0.8em'),
            *row_cells))

    return Div(
        Card(
            Table(Thead(header), Tbody(*rows),
                  cls=TableT.sm,
                  style='border-collapse:collapse'),
            header=CardTitle(title),
            style='overflow-x:auto',
        ),
        _role_legend(),
    )

In [ ]:
#space_time_timetable(plan.schedule, corridor=corridor, grid=myStuff.grid)

How did we have so many pieces at the same hex at the start

Is there anything we should refactor?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good moment to step back and look at the whole picture. Here are the spots I'd prioritize:

**1. `space_time_timetable` piece→role matching is a mess**

The function tries three different strategies to map `piece_id` → `(emoji, color)` and the later one silently overwrites the earlier. The `role_names` dict is built but never read. The queen-detection branch has a confusing double-negative comprehension. I'd extract a clean helper:

```python
def _piece_role_map(df, roster=None):
    """Map piece_id → role string using roster order (shadows match roster 1:1)."""
    ids = list(df.drop_duplicates('piece_id').piece_id)
    if roster and len(roster) == len(ids):
        return {pid: r['role'] for pid, r in zip(ids, roster)}
    # Fallback: pattern-match on name
    patterns = [('camp_o','origin_camp'),('camp_d','advance_party'),
                ('station','station'),('blh_','lighthouse'),('besc_','escort')]
    out = {}
    for _, row in df.drop_duplicates('piece_id').iterrows():
        name = str(row.piece_name).lower()
        out[row.piece_id] = next((role for pat,role in patterns if pat in name), 'queen')
    return out
```

Then `space_time_timetable` just calls `roles = _piece_role_map(df, roster)` and does one lookup.

**2. `_hex_sort_key` is defined but unused** — the actual sorting lives inline inside `space_time_timetable`. Kill the dead function or use it.

**3. `QueenCoverage` has too many jobs**

It builds shadows (4 separate `_build_*` methods), plans camps, constructs corridors, *and* orchestrates simulation retries. The roster-assembly block inside `propose_crossing` is a natural seam — extract a `RosterBuilder` or just a plain function `build_crossing_roster(camp, corridor, queen, profile) → (shadows, roster, queen_start)` so you can test roster logic without running a sim.

**4. `SimAdapter._map_loc` looks broken**

```python
w = cor.index(hex_idx)    # returns int (route index)
seg = cor[w]              # Corridor doesn't support __getitem__ on route index?
```

Unless `Corridor.__getitem__` resolves a route index to a segment, this will error or return wrong data. Worth a unit test.

**5. Magic numbers**

- `* 60` and `* 80` for park instruction repetitions
- `attempt * 2` extra wait
- `math.ceil(len(bp) / 3) + 2` bishop arrival estimate

These should be named constants or derived from the profile, e.g. `PARK_TURNS = 60`, `BISHOP_SPEED = 3`.

**6. Duplicate imports across cells** — `dataclass`, `field`, `heapq`, `math`, `pd` are imported 2-3 times. Not a runtime issue but noisy when reading.

I'd start with **#1** (the timetable role map) since it's self-contained and immediately cleans up the messiest function, then tackle **#3** (extracting roster assembly) since that's where the most complexity lives.

How would we get two queens across the board. perhaps starting from different locations. The more general part is that we have something that works for a single target. how do we do multiple sources and sinks

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great question — this is the classic jump from "plan one trip" to "plan a logistics network". There are three broad approaches, each with different trade-offs:

**A. Independent plans + deconfliction** — Run `propose_crossing` per queen, then resolve resource conflicts after (e.g. "both plans want the same bishop"). Simplest to build, but can produce suboptimal or infeasible results when resources are tight.

**B. Shared corridor network** — Build the corridor graph first (all interesting source→sink edges), then *schedule* queens through it, sharing stations and bishops where routes overlap. This is the "build roads, then plan trips" model.

**C. Joint optimization** — One big MIP that allocates pawns, bishops, corridor segments, and timing across all queens simultaneously. Most powerful but hardest to formulate and slowest to solve.

The practical sweet spot for most strategy games is **B** — it separates *infrastructure* (which changes slowly) from *scheduling* (which changes per turn). Your `QueenCoverage` already owns the corridor list, so the main new piece would be a **scheduler** that takes multiple `(queen, source, sink)` requests and assigns shared resources across them.

The first design fork is: **can two queens share the same corridor segment simultaneously**, or do they need to be sequenced? That determines whether the scheduler is a simple allocation problem or a time-slotted one.

What's your instinct on that — shared or sequenced?

let talk about the B. Shared corridor network — Build the corridor graph first (all interesting source→sink edges). can we do the map with a few interesting postions. There might be another river valley we could get as a pawn farm

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice — so the first step for B is identifying the **nodes** of the corridor graph before we worry about edges.

You already have two natural nodes: the cities. For pawn farms, you'd want clusters of high food-tier hexes — especially river valleys where tier 5-7 hexes clump together. Those are your "sources" in the network.

A quick way to find candidates: scan for hexes where `fy.tiers >= 5` and cluster them spatially (e.g. connected components of high-tier hexes), then pick the centroid of each cluster as a candidate node.

Want to start by just plotting all the tier 5+ hexes on the map to eyeball where the good valleys are? Something like:

```python
hot = [i for i,t in enumerate(fy.tiers) if t >= 5 and myStuff.terr.elevations[i] > 0]
print(f"{len(hot)} high-tier hexes")
```

That'll give you the raw count, then we can visualize and pick the interesting ones.

In [ ]:
??GameParts

In [ ]:
from HexMagic.climate import ClimatePreset, Climate, TerraDemo, TerrainFactory

In [ ]:
world = TerrainFactory.create_world(
                bounds= MapRect(MapCord(0, 0), MapSize(400, 400)),
                preset='temperate',
                name='Maiden Lane',
                radius=15,
                lon_span=10.0,
                num_plates=8,
                subdivisions=3,
                ocean_fraction=0.3,
                oceanic_sides=['S'],  # Ocean on east and west
                terrain_age='young',  # Sharp, dramatic features
                formation_type='ridge',  # Creates ridge formations
                elevation_scale=1,  # Exaggerate the heights
                erosion_age=0.5,  # Minimal erosion for sharp peaks
                num_lakes=0,
                seed=23,
                debug=True
            )

In [ ]:
myStuff.loadGeology(world)

In [ ]:
ctx = myStuff.overlayContext()

TerrainDisplay(
    SettlementOverlay(),
    RiverOverlay(),
    terrain=myStuff.terr,
    board=myStuff.board,
    basins=myStuff.basin,
    context_cls=GameContext,
    #debug = not showDemo
)

In [ ]:
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

cities = []
for country in myStuff.board.kingdoms:
    for city in country.settlements:
        cities.append(city)

c0, c1 = cities[0].location, cities[1].location
print(f"Cities: ⬡{c0} → ⬡{c1}\n")

# 2. Build coverage + corridor
qc = QueenCoverage(
    board=myStuff.board, grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers, profile=FOOD_PROFILE_V2,
    queens=[myStuff.queen],
)

corridor = qc.build_corridor(c0, c1, kind=CorridorType.PERMANENT,
                              n_pawns=10, n_bishops=4)
print(f"\nCorridor: {corridor.length} hexes, "
      f"{corridor.n_stations} stations, {corridor.n_gaps} gaps, "
      f"feasible={'✅' if corridor.feasible else '❌'}")

if not corridor.feasible:
    print("❌ Corridor not feasible — can't test crossing")
else:
    # 3. Teleport queen to corridor start, fill her up
    queen = myStuff.queen
    queen.location = corridor.hexes[0]
    apply_food_profile([queen])
    print(f"\nQueen {queen.name} at ⬡{queen.location}, "
          f"food={queen.food:.1f}, diet={queen.diet}")

    # 4. Propose crossing
    print(f"\n{'='*60}")
    print("Proposing crossing plan...")
    plan = qc.propose_crossing(
        corridor, queen,
        available_pawns=5,
        available_bishops=2,
        n_turns=60,
        max_retries=3,
    )

    # 5. Results
    print(f"\n{'='*60}")
    print(f"Feasible:     {plan.feasible}")
    print(f"Pieces:       {dict(plan.pieces_needed())}")
    print(f"Queen starts: turn {plan.queen_start_turn}")
    print(f"Arrival:      turn {plan.queen_arrival}")
    print(f"Food margin:  {plan.food_margin:.1f}")
    print(f"Bishop fuel:  {plan.bishop_fuel_used:.0f}")

    # 6. Show the plan card + timetable
    show(plan)

Why doesn't this work?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The corridor is longer now (25 hexes vs 19 before), and the critical clue is **bishop fuel used: 7** (out of 280 available). The bishops barely did anything, which means the queen likely never reached the gap segments.

The root cause: your **advance party pawns start at route[0] (⬡492) and need to sprint all the way to hexes 162 and 185** near the destination. That's roughly the full 25-hex corridor length. Pawns have only 12 food capacity and eat 2/turn — they have about **6 turns of autonomy**. They starve long before reaching their destination camps.

So the far-end food infrastructure never gets established, and the queen has no support for the final 6-hex caravan gap (which alone costs her 21 food at diet=7).

The first test worked because the corridor was short enough that advance pawns could reach the far end before running out of food. This one is too long for a single sprint.

A few ways to fix it:
1. **Stage the advance party** — leapfrog them through station segments so they refuel en route
2. **Use local pawns** — recruit dest-camp pawns from near the destination instead of sending them from the origin
3. **Add more station segments** — so no sprint exceeds pawn autonomy range

Which direction feels most natural for your game design?

so what we want is to find the max feeding station we can get to with sprinting pawns.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Right — you need a **pawn sprint range** function. Walk the route from the origin, decrement food each step, optionally harvest if the hex has food, and see how far the pawn gets before hitting zero. Something like:

- Start at `food = pawn_capacity` (12)
- Each step: `food -= pawn_diet` (2), then if current hex has food tier > 0, `food += min(harvest, tier_yield)`, capped at capacity
- Stop when `food <= 0` — that's max reach

Then compare that against the station positions in the corridor. Any station beyond max reach needs to be supplied differently (local recruit, relay, or an extra station closer in).

Want to write that as a helper on `QueenCoverage`, or a standalone function?

What I am thinking is we want a list of available feeding pawn stations on the map available on the map. the user can select one to target to set up a feeding statition. next given a cover of feeding plan stations you can move pieces like queens to them.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

That's a really clean two-layer design:

1. **Station discovery** — find all viable pawn-farm locations on the map (high food-tier clusters)
2. **Station selection** — player picks which ones to establish
3. **Queen routing** — plan queen movement as a chain of station hops

The key constraint linking 1→2 is **reachability**: a station is only useful if you can get pawns to it from an existing station (or a city) within their sprint range.

So the data you'd want for each candidate station is:
- **Location** (hex or cluster centroid)
- **Yield** (total food export if staffed)
- **Reachable from** — which existing stations/cities can sprint a pawn there
- **Sprint cost** — how much food it takes to get a pawn there

That naturally gives you a graph: cities and stations are nodes, edges exist where pawn sprint range allows.

Does that match your mental model, or are you thinking of it differently?

the graph model works, but it has interesting edges. you have basic sprint length and then length supported by a bishop.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Exactly — so each edge in the graph has two reach values:

- **Pawn-only range** — ~6 hexes through barren, more if there's food along the way
- **Bishop-assisted range** — much further, since the bishop covers the barren gap (your existing `bishop_capacity / (bishop_diet + queen_diet)` math)

And the edge cost changes accordingly:
- Pawn-only edge: costs N pawns, zero bishops
- Bishop-assisted edge: costs N pawns + 1 bishop, and has a **duration** (bishop fuel is the clock)

So you're really building a **resource-annotated graph** where edges carry `(pawns_needed, bishops_needed, turns, max_gap)`. Then "can I get queen A from city X to station Y" becomes a shortest-path problem where the budget constraint is your available pieces.

Does the bishop always escort, or can it also do the lighthouse (stationary overwatch) role on these edges?

Yes developing a resource-annoated graph is what we want. I open to an strategy about how to get pieces across it.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great. Let's build this in layers. The first thing we need is the **node type**:

```python
@dataclass
class LogisticsNode:
    hex: int
    kind: str          # 'city', 'station', 'outpost'
    yield_per_turn: float
    staffing: int      # pawns needed to run it
```

Then edges:

```python
@dataclass
class LogisticsEdge:
    route: list[int]
    gap_hexes: int
    pawn_only: bool    # reachable by pawn sprint alone?
    bishops: int       # bishops needed if not pawn-only
    cost_turns: int    # transit time for queen
```

The graph is then `dict[node_id, list[(node_id, LogisticsEdge)]]`.

Does this feel like the right granularity for the node/edge data, or do you want to capture more (e.g. bishop fuel burn per edge, or which specific hexes are stations)?